# Validación cruzada y búsqueda de hiperparámetros

[Validación y selección de modelos](validation_and_model_selection.ipynb) apartó un conjunto de validación para comparar modelos. Funciona, con dos costes: las filas de validación son filas de las que el modelo no aprende, y el error de validación se mide con una sola muestra de filas, así que se mueve cuando cambia la muestra. La **validación cruzada** reduce los dos costes por turnos: cada fila se usa para validar una vez, y para entrenar el resto del tiempo. Con ella, los ajustes de un modelo se pueden buscar de forma sistemática.

## Por turnos

En la **validación cruzada de *k* pliegues** (*k-fold*), las filas de entrenamiento se dividen en *k* partes del mismo tamaño, los *pliegues*. El modelo se entrena *k* veces, cada vez con todos los pliegues menos uno, y se mide con el que queda fuera. Con 5 pliegues:

| Ronda | Pliegue 1 | Pliegue 2 | Pliegue 3 | Pliegue 4 | Pliegue 5 |
| --- | --- | --- | --- | --- | --- |
| 1 | **validación** | entrenamiento | entrenamiento | entrenamiento | entrenamiento |
| 2 | entrenamiento | **validación** | entrenamiento | entrenamiento | entrenamiento |
| 3 | entrenamiento | entrenamiento | **validación** | entrenamiento | entrenamiento |
| 4 | entrenamiento | entrenamiento | entrenamiento | **validación** | entrenamiento |
| 5 | entrenamiento | entrenamiento | entrenamiento | entrenamiento | **validación** |

El resultado son *k* errores de validación en lugar de uno: su media es una estimación más estable, y su dispersión muestra cuánto se movería con otra muestra. El coste son *k* entrenamientos en lugar de uno. El conjunto de prueba se queda fuera de todo esto.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True).frame
X = housing.drop(columns="MedHouseVal")
y = 100_000 * housing["MedHouseVal"]
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
len(X_rest), len(X_test)

(16512, 4128)

Esta vez no se separa un conjunto de validación: la validación cruzada hace sus propios pliegues con las 16 512 filas que no son el conjunto de prueba. Con el modelo de gradient boosting de [Gradient boosting](../algorithms/tree-based/gradient_boosting.ipynb):

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score

folds = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(HistGradientBoostingRegressor(random_state=42), X_rest, y_rest, cv=folds,
                         scoring="neg_mean_absolute_error")
scores

array([-33093.633457  , -31298.82743634, -31377.57478989, -31397.99704353,
       -31083.76955568])

Las puntuaciones son negativas porque las búsquedas de scikit-learn siempre *maximizan* una puntuación, así que los errores se dan como utilidades, con el signo cambiado: cuanto mayor (más cerca de 0), mejor. Volviéndolos a convertir en errores:

In [ ]:
mae_by_fold = -scores
mae_by_fold.round(0), mae_by_fold.mean().round(0), mae_by_fold.std().round(0)

(array([33094., 31299., 31378., 31398., 31084.]),
 np.float64(31650.0),
 np.float64(730.0))

Los cinco pliegues dan errores de entre unos 31 100 \$ y 33 100 \$, con una media de 31 700 \$ y una dispersión (desviación típica) de unos 730 \$. Un solo conjunto de validación habría dado uno de estos números, y cuál habría dependido de la división. La media es más estable, y la dispersión es una vara de medir: una diferencia entre dos modelos menor de unos 1000 \$ podría ser casualidad.

## Pliegues que encajan con el problema

El `shuffle=True` de arriba no era un detalle. Sin él, `KFold` corta las filas en el orden en que están guardadas, y estas filas están guardadas condado a condado: las primeras en Alameda, en el área de la bahía, las últimas en Yuba, en el norte.

In [ ]:
ordered_folds = KFold(n_splits=5)
-cross_val_score(HistGradientBoostingRegressor(random_state=42), X_rest.sort_index(), y_rest.sort_index(), cv=ordered_folds,
                 scoring="neg_mean_absolute_error").round(0)

array([45263., 40782., 44164., 44656., 46561.])

Sin barajar, el error salta a entre 40 800 \$ y 46 600 \$. Cada pliegue es ahora un bloque de condados enteros, y cada ronda pide al modelo predecir condados que nunca ha visto, sin un solo distrito de esos condados entre sus filas de entrenamiento. Es una pregunta mucho más difícil que predecir un distrito cuyos vecinos estaban en los datos de entrenamiento, y no es una pregunta equivocada: si el modelo se fuera a usar en condados nuevos, estos serían los pliegues correctos, y los barajados serían optimistas. Los pliegues no solo miden el modelo; definen qué se está midiendo.

Los pliegues deberían copiar la división que encaja con la producción, como explicaba [el conjunto de prueba representa la producción](overfitting_and_the_split.ipynb#el-conjunto-de-prueba-representa-la-producción), y scikit-learn tiene un divisor para cada caso:

| Lo que traerá la producción | Divisor | Qué hace |
| --- | --- | --- |
| filas nuevas como las antiguas | `KFold(shuffle=True)` | pliegues aleatorios |
| lo mismo, para clasificación | `StratifiedKFold(shuffle=True)` | pliegues aleatorios con la misma proporción de cada clase |
| el futuro | `TimeSeriesSplit` | cada ronda entrena con las filas anteriores a un momento y valida con las filas justo posteriores |
| entidades nuevas: pacientes, casas, lugares | `GroupKFold` | todas las filas de una entidad en el mismo pliegue |

Un entero como `cv=5` significa `KFold`, o `StratifiedKFold` para un clasificador, *sin* barajar: pasa un divisor siempre que el orden de las filas signifique algo.

`TimeSeriesSplit`, con doce filas ordenadas en el tiempo, muestra sus rondas: las filas de entrenamiento siempre van antes que las de validación.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

pd.DataFrame([
    {"training rows": f"{training[0]}–{training[-1]}", "validation rows": f"{validation[0]}–{validation[-1]}"}
    for training, validation in TimeSeriesSplit(n_splits=4).split(np.arange(12))
], index=pd.RangeIndex(1, 5, name="round"))

,training rows,validation rows
round,,
1,0–3,4–5
2,0–5,6–7
3,0–7,8–9
4,0–9,10–11


## Hiperparámetros

Los **parámetros** de un modelo los aprende `fit`: los pesos de un modelo lineal, las divisiones de un árbol. Sus **hiperparámetros** se eligen antes de `fit`: la tasa de aprendizaje, el tamaño de los árboles, el número de vecinos, la penalización. [Controlar la complejidad](regularization.ipynb) mostró que cada modelo tiene algunos, y que los mejores valores dependen de los datos. Probar valores a mano y mirar los errores es lento y fácil de sesgar; una **búsqueda** los prueba de forma sistemática, cada uno medido por validación cruzada.

## Búsqueda en rejilla

`GridSearchCV` prueba todas las combinaciones de los valores enumerados. Tres tasas de aprendizaje y tres tamaños de árbol son nueve combinaciones, cada una medida con 5 pliegues: 45 entrenamientos. `n_jobs=-1` los ejecuta en paralelo en todos los núcleos:

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    HistGradientBoostingRegressor(max_iter=300, random_state=42),
    param_grid={"learning_rate": [0.03, 0.1, 0.3], "max_leaf_nodes": [15, 31, 63]},
    cv=folds, scoring="neg_mean_absolute_error", n_jobs=-1,
).fit(X_rest, y_rest)
grid_search.best_params_, -grid_search.best_score_

({'learning_rate': 0.1, 'max_leaf_nodes': 63}, np.float64(30456.196361613507))

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
results.assign(mean_MAE=-results["mean_test_score"]).pivot(
    index="param_learning_rate", columns="param_max_leaf_nodes", values="mean_MAE").round(0)

param_max_leaf_nodes,15,31,63
param_learning_rate,,,
0.03,33573.0,31622.0,30638.0
0.10,31554.0,30735.0,30456.0
0.30,32200.0,32147.0,31799.0


La mejor combinación es una tasa de aprendizaje de 0,1 con árboles de hasta 63 hojas, con unos 30 460 \$, y los árboles más grandes ganan con todas las tasas de aprendizaje. 63 es el tamaño más grande de la rejilla, así que una segunda búsqueda debería probar también árboles más grandes, como se explica más abajo. Dejando fuera la más lenta (una tasa de aprendizaje de 0,03 con 15 hojas), las combinaciones están a menos de unos 1750 \$ entre sí, entre dos y tres veces la dispersión entre pliegues medida arriba.

`n_jobs` merece cuidado: la búsqueda ejecuta entrenamientos en paralelo, y algunos modelos también paralelizan dentro de cada entrenamiento. Las dos cosas a la vez multiplican los hilos que compiten por los mismos núcleos. Lo habitual es paralelizar la búsqueda y dejar al modelo un solo núcleo.

## Búsqueda aleatoria

Una rejilla crece rápido: cuatro hiperparámetros con cinco valores cada uno son 625 combinaciones, 3125 entrenamientos con 5 pliegues. `RandomizedSearchCV` prueba en cambio un número fijo de combinaciones aleatorias, sacadas de distribuciones. Su presupuesto no crece con el número de hiperparámetros, y como normalmente solo unos pocos de ellos importan mucho, las combinaciones aleatorias prueban más valores distintos de esos pocos que una rejilla del mismo tamaño. Los ajustes continuos, como una tasa de aprendizaje, se sacan de un rango continuo en lugar de unos pocos puntos elegidos; `loguniform` reparte las extracciones por igual entre órdenes de magnitud, para que de 0,01 a 0,03 reciba tantos intentos como de 0,1 a 0,3:

In [ ]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    HistGradientBoostingRegressor(max_iter=300, random_state=42),
    param_distributions={
        "learning_rate": loguniform(0.01, 0.3),
        "max_leaf_nodes": randint(8, 128),
        "min_samples_leaf": randint(5, 100),
        "l2_regularization": loguniform(1e-3, 10),
    },
    n_iter=20, cv=folds, scoring="neg_mean_absolute_error", n_jobs=-1, random_state=42,
).fit(X_rest, y_rest)
random_search.best_params_, -random_search.best_score_

({'l2_regularization': np.float64(0.020237789540485076),
  'learning_rate': np.float64(0.06960091021987622),
  'max_leaf_nodes': 69,
  'min_samples_leaf': 44},
 np.float64(30100.92593726995))

Veinte combinaciones aleatorias de cuatro ajustes llegan a unos 30 100 \$, algo mejor que la mejor de la rejilla, con árboles de 69 hojas, una tasa de aprendizaje de 0,07 y hojas más grandes (de al menos 44 filas cada una). La diferencia con la rejilla, unos 360 \$, es la mitad de la dispersión entre pliegues: las dos búsquedas han encontrado regiones de ajustes igual de buenas, no un modelo mejor.

Una primera búsqueda rara vez es la última. Lo habitual es mirar los resultados, fijar los hiperparámetros cuyos mejores valores coinciden, y volver a buscar en un rango más estrecho alrededor de los mejores valores del resto; cuando el mejor valor de un ajuste está en el borde del rango buscado, el rango se amplía en esa dirección.

## Buscar en un pipeline

Una búsqueda puede ajustar cualquier paso de un [pipeline](../basic-workflows/california_with_pipelines.ipynb), con el nombre del paso, dos guiones bajos y el nombre del ajuste. Como el pipeline entero se vuelve a ajustar en cada pliegue, el escalador se ajusta solo con las filas de entrenamiento de ese pliegue, y nada se filtra desde el pliegue de validación: `make_pipeline` nombra cada paso según su clase, en minúsculas, y un ajuste de un paso se escribe como el nombre del paso, dos guiones bajos y el ajuste: `kneighborsregressor__n_neighbors`.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

knn_search = GridSearchCV(
    make_pipeline(StandardScaler(), KNeighborsRegressor()),
    param_grid={"kneighborsregressor__n_neighbors": [5, 10, 20, 40], "kneighborsregressor__weights": ["uniform", "distance"]},
    cv=folds, scoring="neg_mean_absolute_error", n_jobs=-1,
).fit(X_rest, y_rest)
knn_search.best_params_, -knn_search.best_score_

({'kneighborsregressor__n_neighbors': 10,
  'kneighborsregressor__weights': 'distance'},
 np.float64(43752.24132233467))

El mejor k-NN usa 10 vecinos ponderados por su distancia, con unos 43 750 \$: muy por detrás del gradient boosting en esta tabla, pero encontrado con la misma maquinaria, escalador incluido.

## La prueba final, una sola vez

Tras la búsqueda, `best_estimator_` es la mejor combinación ya reentrenada con todas las filas que recibió la búsqueda. Se mide una sola vez con el conjunto de prueba:

In [ ]:
from sklearn.metrics import mean_absolute_error

{
    "grid search": mean_absolute_error(y_test, grid_search.best_estimator_.predict(X_test)),
    "randomised search": mean_absolute_error(y_test, random_search.best_estimator_.predict(X_test)),
}

{'grid search': 29244.879668895162, 'randomised search': 29503.953001311715}

Los dos modelos ajustados fallan los distritos de prueba por entre unos 29 200 \$ y 29 500 \$, el de la rejilla ligeramente por delante, que es lo contrario de su orden en la validación cruzada: diferencias de unos cientos de dólares están dentro del ruido de las dos medidas. Lo que se puede decir es que el ajuste llevó el error de prueba de unos 31 000 \$ con los valores por defecto ([Medir el error de un modelo de regresión](../evaluation/regression_metrics.ipynb), con la misma división) a unos 29 200 \$.

La mejor puntuación de validación cruzada de una búsqueda es optimista por el mismo motivo que en [Validación y selección de modelos](validation_and_model_selection.ipynb#elegir-con-el-conjunto-de-prueba): es la mejor de muchas estimaciones ruidosas. El conjunto de prueba lo corrige. Cuando ni siquiera el conjunto de prueba se puede permitir, la validación cruzada *anidada* ejecuta toda la búsqueda dentro de cada pliegue de una validación cruzada exterior, y estima el error del procedimiento (búsqueda incluida) con un coste mucho mayor.